In [1]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["CUDA_VISIBLE_DEVICES"] = "5"


from datasets import load_dataset, load_from_disk
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer,Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq, pipeline
import evaluate
import numpy as np
from peft import LoraConfig, get_peft_model
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from peft import get_peft_model, LoraConfig, TaskType
from tqdm import tqdm
import json

import torch
print(torch.cuda.device_count())

from typing import List, Dict
import warnings
warnings.filterwarnings('ignore')

import logging as pylogging
from transformers import logging

# Keep normal Trainer progress visible
logging.set_verbosity_warning()

# Silence only LED model padding warnings
pylogging.getLogger("transformers.models.led.modeling_led").setLevel(pylogging.ERROR)

/home/23ucs642/miniconda3/envs/myenv/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/home/23ucs642/miniconda3/envs/myenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1


In [ ]:
dataset = load_from_disk("/home/23ucs642/Video_Summarization_Translation/tib_dataset")

train_dataset = dataset["train"]
valid_dataset = dataset["valid"]

In [3]:
train_dataset[0]

{'doi': '10.5446/18349 (DOI)',
 'title': 'Modern Steel Products (2014) - Wire & Rod Mills: lecture 23',
 'url': 'https://av.tib.eu/media/18349',
 'video_url': 'https://tib.flowcenter.de/mfc/medialink/3/deb95d46836b0e23c09a8940e8a11d4f391e1a5bae1468c9d3f3cc82a5815fc65c5c/Modern_Steel_Products_-_2014__Wire___Rod_Mills_23.mp4',
 'license': 'CC Attribution 3.0 Unported:\nYou are free to use, adapt and copy, distribute and transmit the work or content in adapted or unchanged form for any legal purpose as long as the work is attributed to the author in the manner specified by the author or licensor.',
 'subject': 'Engineering',
 'genre': 'Lecture',
 'release_year': '2014',
 'author': 'Cooman, Bruno C. de',
 'contributors': None,
 'abstract': 'Professor de Cooman takes the topic of rod and bar manufacture, including non-destructive testing. Both the production and metallurgy of the alloys is described. This is a part of a course of lectures given at the Graduate Institute of Ferrous Technolog

In [4]:
model_name = "allenai/led-base-16384"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
base_model = model

In [5]:
print("Enabling gradient checkpointing...")
model.gradient_checkpointing_enable()
model.config.use_cache = False

Enabling gradient checkpointing...


In [13]:
rouge_metric = evaluate.load("rouge")
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    predictions = np.array(predictions)
    labels = np.array(labels)

    predictions[predictions < 0] = tokenizer.pad_token_id

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    result = {k: round(v * 100, 2) for k, v in result.items()}
    return result

In [15]:
model_name = "allenai/led-base-16384"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
base_model = model

In [16]:
MAX_INPUT = 8192 
MAX_TARGET = 512

def preprocess_function(batch):
    inputs = batch["transcript"]
    targets = batch["abstract"]
    model_inputs = tokenizer(
        inputs, max_length=MAX_INPUT, truncation=True, padding="max_length"
    )
    labels = tokenizer(
        targets, max_length=MAX_TARGET, truncation=True, padding="max_length"
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing...")
tokenized = dataset.map(preprocess_function, batched=True, remove_columns=dataset["train"].column_names)

train_ds = tokenized["train"]
val_ds = tokenized["valid"]

Tokenizing...


In [17]:
for param in model.parameters():
    param.requires_grad = False

In [18]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=32,                  
    lora_alpha=32,       
    lora_dropout=0.1,   
    bias="none",      
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 1,179,648 || all params: 163,024,128 || trainable%: 0.7236


In [19]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

metric = evaluate.load("rouge")

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    result = {k: round(v * 100, 4) for k, v in result.items()}
    return result

In [20]:
training_args = Seq2SeqTrainingArguments(
    output_dir="/home/23ucs642/Video_Summarization_Translation/led_tib_lora",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    predict_with_generate=True,
    generation_max_length=512,
    learning_rate=6e-4,
    num_train_epochs=20,
    eval_strategy="steps",
    eval_steps=200,
    save_steps=200,
    logging_steps=200,
    save_total_limit=2,
    report_to="none",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",   
    greater_is_better=False,    
    save_strategy="steps",
    fp16=True,
    dataloader_num_workers=4,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print("Starting training...")
trainer.train()

print("Saving LoRA adapter weights...")
model.save_pretrained("/home/23ucs642/Video_Summarization_Translation/led_tib_lora")
tokenizer.save_pretrained("/home/23ucs642/Video_Summarization_Translation/led_tib_lora")

Detected kernel version 3.10.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Starting training...


Step,Training Loss,Validation Loss
200,5.723900,4.417972
400,4.632800,4.385523
600,4.608900,4.377080
800,4.582200,4.366163
1000,4.574600,4.365840
1200,4.558600,4.360623
1400,4.561000,4.352574
1600,4.544800,4.352836
1800,4.544900,4.345337
2000,4.538000,4.341613


Saving LoRA adapter weights...


('/home/23ucs642/Video_Summarization_Translation/led_tib_lora/tokenizer_config.json',
 '/home/23ucs642/Video_Summarization_Translation/led_tib_lora/special_tokens_map.json',
 '/home/23ucs642/Video_Summarization_Translation/led_tib_lora/vocab.json',
 '/home/23ucs642/Video_Summarization_Translation/led_tib_lora/merges.txt',
 '/home/23ucs642/Video_Summarization_Translation/led_tib_lora/added_tokens.json',
 '/home/23ucs642/Video_Summarization_Translation/led_tib_lora/tokenizer.json')